In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName('RDD_Operations')\
.master('yarn')\
.getOrCreate()

26/06/02 14:55:17 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
customer_data = [
    "customer_id,name,city,state,country,registration_date,is_active",
    "0,Customer_0,Bangalore,Karnataka,India,2023-11-11,True",
    "1,Customer_1,Hyderabad,Delhi,India,2023-08-26,True",
    "2,Customer_2,Ahmedabad,West Bengal,India,2023-06-23,True",
    "3,Customer_3,Bangalore,Tamil Nadu,India,2023-03-24,False",
    "4,Customer_4,Bangalore,Gujarat,India,2023-06-06,False",
    "5,Customer_5,Delhi,Maharashtra,India,2023-04-19,False"
]

# RDD - RESILIENT DISTRIBUTED DATASET

In [4]:
data_rdd = spark.sparkContext.parallelize(customer_data)

In [5]:
data_rdd.getNumPartitions()

2

# first() - gives first row or element of RDD

In [6]:
header = data_rdd.first()

In [7]:
header

'customer_id,name,city,state,country,registration_date,is_active'

# Filter() - based on the condition, its going to transform dataset

In [8]:
data_rdd = data_rdd.filter(lambda row : row !=header)

In [10]:
data_rdd.collect()

['0,Customer_0,Bangalore,Karnataka,India,2023-11-11,True',
 '1,Customer_1,Hyderabad,Delhi,India,2023-08-26,True',
 '2,Customer_2,Ahmedabad,West Bengal,India,2023-06-23,True',
 '3,Customer_3,Bangalore,Tamil Nadu,India,2023-03-24,False',
 '4,Customer_4,Bangalore,Gujarat,India,2023-06-06,False',
 '5,Customer_5,Delhi,Maharashtra,India,2023-04-19,False']

# Map() - it applies a function to each element(row) in rdd` 

In [14]:
def parse_row(row):
    fields = row.split(',')
    return(
        int(fields[0]),
            fields[1],
            fields[2],
            fields[3],
            fields[4],
            fields[5],
            fields[6]=='True')

In [16]:
parsed_rdd = data_rdd.map(parse_row)

In [17]:
parsed_rdd.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True),
 (3, 'Customer_3', 'Bangalore', 'Tamil Nadu', 'India', '2023-03-24', False),
 (4, 'Customer_4', 'Bangalore', 'Gujarat', 'India', '2023-06-06', False),
 (5, 'Customer_5', 'Delhi', 'Maharashtra', 'India', '2023-04-19', False)]

### Advance RDD Operations

In [18]:
#extract a field with map() -city and state


In [19]:
name_city_rdd = parsed_rdd.map(lambda row: (row[1],row[2]))

In [20]:
name_city_rdd .collect()

[('Customer_0', 'Bangalore'),
 ('Customer_1', 'Hyderabad'),
 ('Customer_2', 'Ahmedabad'),
 ('Customer_3', 'Bangalore'),
 ('Customer_4', 'Bangalore'),
 ('Customer_5', 'Delhi')]

In [21]:
#filter out active customers

In [23]:
active_customers= parsed_rdd.filter(lambda row :row[6]==True)

In [25]:
active_customers.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True)]

In [26]:
# distinct() - Transformation

In [27]:
cities = parsed_rdd.map(lambda row: row[2]).distinct()

In [28]:
cities.collect()

['Hyderabad', 'Delhi', 'Bangalore', 'Ahmedabad']

In [31]:
#take()

cities.take(3)

['Hyderabad', 'Delhi', 'Bangalore']

In [33]:
#reduceByKey() -Transformation - combines values for each key using associative reduce function

In [36]:
citiesPerCust = parsed_rdd.map(lambda row : (row[2],1)).reduceByKey(lambda a,b:a+b)

In [38]:
citiesPerCust.collect()

[('Hyderabad', 1), ('Delhi', 1), ('Bangalore', 3), ('Ahmedabad', 1)]

In [39]:
#count by value

In [42]:
count_rdd=parsed_rdd.map(lambda row: row[2]).countByValue()

In [44]:
count_rdd

defaultdict(int, {'Bangalore': 3, 'Hyderabad': 1, 'Ahmedabad': 1, 'Delhi': 1})

### reduce by key is a transformation and count by value is an action

In [51]:
#combine more operations

active_cities = parsed_rdd.filter(lambda row: row[6]) \
                          .map(lambda row :row[2]) \
                          .distinct()


In [52]:
active_cities.collect()

['Hyderabad', 'Bangalore', 'Ahmedabad']

In [53]:
#count active customers by state

In [65]:
custByState = parsed_rdd.filter(lambda row : row[6]=='True') \
                        .map(lambda row: (row[3],1))\
                        .reduceByKey(lambda a,b:a+b)

In [64]:
custByState.collect()

[]

In [66]:
parsed_rdd.take(10)

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True),
 (3, 'Customer_3', 'Bangalore', 'Tamil Nadu', 'India', '2023-03-24', False),
 (4, 'Customer_4', 'Bangalore', 'Gujarat', 'India', '2023-06-06', False),
 (5, 'Customer_5', 'Delhi', 'Maharashtra', 'India', '2023-04-19', False)]